# Article 50 Watermark Pilot — Gemma 2 9B

This notebook runs the approved **20-prompt engineering pilot** on a no-cost Colab GPU. It uses the frozen model revision, seeds, split, generation settings, and tripled calibration controls. Pilot outputs are quarantined and are not confirmatory evidence.

Success means every planned generation has either a cached output or a recorded failure, both detectors score the outputs, and the artifact bundle is copied off the ephemeral runtime. **Ali must personally complete the Hugging Face login cell.**

## 1. Runtime and frozen configuration

In Colab choose **Runtime → Change runtime type → GPU**. An L4/A100 is preferable; a T4 may not hold the pinned 9B model without offload. Quantization and model substitution are protocol changes and are not enabled here.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, torch
assert torch.cuda.is_available(), 'Stop: select a GPU runtime before continuing.'
gpu = torch.cuda.get_device_properties(0)
runtime = {'gpu': gpu.name, 'memory_gib': round(gpu.total_memory / 2**30, 2), 'python': sys.version}
print(json.dumps(runtime, indent=2))

## 2. Fetch the draft branch and install exact dependencies

The preregistration freeze commit must remain in the branch history. The notebook records the actual checked-out commit in every run manifest.

In [ ]:
REPO_URL = 'https://github.com/AliHasan-786/llm-invisible-watermarking.git'
BRANCH = 'codex/watermark-p0'
WORKDIR = pathlib.Path('/content/llm-invisible-watermarking')
if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
freeze = subprocess.check_output(['git', 'log', '-1', '--format=%H', '--', 'PREREGISTRATION.md'], text=True).strip()
assert freeze.startswith('0869e396'), f'Unexpected preregistration freeze: {freeze}'
print({'head': head, 'preregistration_freeze': freeze})

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

## 3. Personal credential gate

Run this cell yourself and paste a read-only Hugging Face token that already has access to `google/gemma-2-9b-it`. The token widget masks input. The token is not written to the repository or result bundle.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # MANUAL: Ali completes this credential gate.

## 4. Freeze prompts and run the quarantined pilot

This creates 20 source-balanced prompts. Calibration prompts receive three control generations; held-out prompts receive matched control, Kirchenbauer, and SynthID generations.

In [ ]:
subprocess.run([sys.executable, 'scripts/run_article50.py', 'preflight', '--model', 'gemma'], check=True)
subprocess.run([sys.executable, 'scripts/run_article50.py', 'freeze-prompts', '--model', 'gemma'], check=True)
subprocess.run([sys.executable, 'scripts/run_article50.py', 'generate', '--model', 'gemma', '--pilot-prompts', '20'], check=True)

## 5. Score clean outputs and deterministic attacks

These are pilot diagnostics only. The amended calibration summary includes realized FPR and a prompt-clustered bootstrap interval.

In [ ]:
pilot = pathlib.Path('results/article50/pilot/gemma')
completions = pilot / 'completions.jsonl'
subprocess.run([sys.executable, 'scripts/score_article50.py', str(completions), '--detector', 'kirchenbauer'], check=True)
subprocess.run([sys.executable, 'scripts/score_article50.py', str(completions), '--detector', 'synthid'], check=True)
subprocess.run([sys.executable, 'scripts/run_article50_attacks.py', str(completions), '--model', 'gemma', '--mode', 'deterministic'], check=True)
print('Pilot generation, clean scoring, and deterministic attacks complete.')

## 6. Completeness check and artifact bundle

The check below does not hide failures: it requires every plan key to appear in either the completion ledger or failure ledger. Download the resulting ZIP before the Colab runtime expires.

In [ ]:
def read_jsonl(path):
    if not pathlib.Path(path).exists(): return []
    return [json.loads(line) for line in pathlib.Path(path).read_text().splitlines() if line.strip()]
def key(row): return (row['prompt_id'], row['split'], row['scheme'], int(row['replicate']))
plan = read_jsonl(pilot / 'generation_plan.jsonl')
done = read_jsonl(pilot / 'completions.jsonl')
failed = read_jsonl(pilot / 'failures.jsonl')
missing = {key(row) for row in plan} - {key(row) for row in done + failed}
assert not missing, f'Missing {len(missing)} planned generations'
summary = {'planned': len(plan), 'completed': len(done), 'failed': len(failed), 'missing': len(missing)}
print(json.dumps(summary, indent=2))
archive = shutil.make_archive('/content/article50_gemma_pilot', 'zip', pilot)
print('Download:', archive)

## Decision after the pilot

- **Continue unchanged** only if the exact pinned model loads and the ledger reaches zero missing generations.
- **Stop and brief** if memory requires quantization, a model/revision substitution, or any frozen protocol change.
- Do not treat pilot TPR as a confirmatory result. Paid compute remains a separate Ali gate.